In [2]:
import numpy as np


a = np.random.rand(2,3,2)
b = np.random.rand(5,4,3)
c = np.random.rand(2,5,3)

a_shape = a.shape[:-1]
b_shape = b.shape[:-1]

res = np.einsum('ai,bj,ikj->abk',a.reshape(-1, a.shape[-1]),b.reshape(-1, b.shape[-1]),c)
res = res.reshape(a_shape + b_shape + (-1,))
res.shape

(2, 3, 5, 4, 5)

In [3]:
from htcross import HTTreeNode, HTTree

tree = HTTree()

tree.root = HTTreeNode(
  [
    HTTreeNode(None, 2, 2, 0, core_init=np.array([[1,2],[7,11]])),
    HTTreeNode(
      [
        HTTreeNode(None, 1, 3, 1, core_init=np.array([[1],[3],[5]])),
        HTTreeNode(None, 1, 4, 2, core_init=np.array([[1],[3],[5],[17]])),
      ],
      r=2,
      core_init=np.random.rand(1,1,2)
    ),    
  ],
  r=1,
  core_init=np.array([[[1], [1]], [[1], [1]]])
)
tree.setup()
print(tree.d)

print(tree)
print(tree[1,:,1:3])


3
Interior node, ranks (2, 2, 1) 
  Leaf node for dim 0 of size 2 
  Interior node, ranks (1, 1, 2) 
    Leaf node for dim 1 of size 3 
    Leaf node for dim 2 of size 4 

[[[ 44.86097145  74.76828574]
  [134.58291434 224.30485723]
  [224.30485723 373.84142872]]]


In [ ]:
a = np.array([[1,2,3],[4,5,6]])
np.tile(a,(2,1))

array([[1, 2, 3],
       [4, 5, 6],
       [1, 2, 3],
       [4, 5, 6]])

In [ ]:
a = np.array([[1,2,3],[4,5,6]])
print(a)
np.repeat(a,2, axis=0)

[[1 2 3]
 [4 5 6]]


array([[1, 2, 3],
       [1, 2, 3],
       [4, 5, 6],
       [4, 5, 6]])

In [ ]:
np.repeat(np.arange(3).reshape(-1,1), 2, axis=0)

array([[0],
       [0],
       [1],
       [1],
       [2],
       [2]])

In [48]:
import numpy as np
from maxvolpy.maxvol import rect_maxvol_svd

a = np.random.rand(10,3)

ind, C = rect_maxvol_svd(a, maxK=3, job='R')
print(C)
print(ind)


np.linalg.solve(a[ind].T, a.T).T

[[ 0.59244083  0.3843576  -0.28791076]
 [ 0.          1.          0.        ]
 [ 0.91032449 -0.50724933  0.55032504]
 [ 0.59318159  0.29433564 -0.02164551]
 [ 1.          0.          0.        ]
 [ 0.58425893  0.17013783  0.26632257]
 [ 0.00184551  0.10170357  0.63853346]
 [ 0.          0.          1.        ]
 [ 0.61573122  0.18507774  0.25890035]
 [ 0.43998146 -0.29773863  0.54821088]]
[4 1 7]


array([[ 5.92440828e-01,  3.84357600e-01, -2.87910765e-01],
       [ 0.00000000e+00,  1.00000000e+00,  0.00000000e+00],
       [ 9.10324485e-01, -5.07249330e-01,  5.50325038e-01],
       [ 5.93181586e-01,  2.94335636e-01, -2.16455101e-02],
       [ 1.00000000e+00, -2.71410387e-17,  0.00000000e+00],
       [ 5.84258935e-01,  1.70137828e-01,  2.66322572e-01],
       [ 1.84551273e-03,  1.01703568e-01,  6.38533456e-01],
       [ 8.73403851e-18,  6.12480952e-17,  1.00000000e+00],
       [ 6.15731222e-01,  1.85077742e-01,  2.58900350e-01],
       [ 4.39981455e-01, -2.97738632e-01,  5.48210876e-01]])

In [ ]:
a = np.tile(np.arange(3).reshape(1,1,3), (2,2,1))
a = np.outer(np.array([-1,1]), np.arange(3))
print(a)
a = np.tile(a.reshape(2,3,1), (1,1,4))

# a = np.swapaxes(a, 0,1)
a = a.reshape(-1,4)
a.flags
a

[[ 0 -1 -2]
 [ 0  1  2]]


array([[ 0,  0,  0,  0],
       [-1, -1, -1, -1],
       [-2, -2, -2, -2],
       [ 0,  0,  0,  0],
       [ 1,  1,  1,  1],
       [ 2,  2,  2,  2]])

In [ ]:
from htcross import HTCrossTreeNode, HTCrossTree
import numpy as np

tree = HTCrossTree()

tree.root = HTCrossTreeNode(
  [
    HTCrossTreeNode(
      [
        HTCrossTreeNode(None, 2, 10, 0),
        HTCrossTreeNode(None, 2, 10, 1),
      ],
      r=2
    ), 
    HTCrossTreeNode(
      [
        HTCrossTreeNode(None, 2, 10, 2),
        HTCrossTreeNode(None, 2, 10, 3),
      ],
      r=2
    ),    
  ],
  r=1
)
# tree.root = HTCrossTreeNode(
#     [
#       HTCrossTreeNode(None, 2, 3, 1),
#       HTCrossTreeNode(None, 2, 2, 0),
#     ],
#     r=1
#   )
tree.setup()
tree.init_randn()
print(tree.shape)

def target_f(inds):
  return 1 / (1 + np.sum(inds, axis=-1))
  # inds = np.moveaxis(inds, -1, 0)
  # return 1 + inds[3] + inds[1]**2 * inds[3]

inds = np.meshgrid(*[np.arange(d, dtype=int) for d in tree.shape], indexing='ij')
inds = np.stack(inds, axis=-1)

target_full = target_f(inds)

ht_full = tree.to_full()
print(f'err = {np.linalg.norm(target_full - ht_full) / np.linalg.norm(target_full):.2e}')

# for 
tree.run_cross(target_f, iterations=20)

ht_full = tree.to_full()
print(f'err = {np.linalg.norm(target_full - ht_full)/ np.linalg.norm(target_full):.2e}')

print(tree)

# print(target_full)
# print(np.round(ht_full, decimals=1))

(10, 10, 10, 10)
err = 2.44e+02
err = 3.43e-01
Interior node, ranks (11, 11, 1) 
  Interior node, ranks (7, 7, 11) 
    Leaf node for dim 0 of size 10 
    Leaf node for dim 1 of size 10 
  Interior node, ranks (3, 7, 11) 
    Leaf node for dim 2 of size 10 
    Leaf node for dim 3 of size 10 



In [6]:
class Parent:
  def __init__(self, a):
    self.a = a

class Child:
  def __init__(self, parent):
    self.parent = parent

tree = Parent(4)

node = Child(tree)
node2 = Child(tree)

print(node.parent.a)
# tree.a = 10

print(node2.parent.a)


4
4


In [ ]:
import numpy as np
ind_a = np.array([1,2,3,4]).reshape(2,2)
ind_b = np.array([5,6,7,8,7,6]).reshape(2,3)
ind = [ind_a, ind_b]

ind_grid = np.meshgrid(*[np.arange(i.shape[0]) for i in ind], indexing='ij')
# print([i[:, grid.flatten()] for i, grid in zip(ind, ind_grid) ])
inds = np.concatenate([i[grid.flatten()] for i, grid in zip(ind, ind_grid) ], axis=-1)

print(inds)
# print(ind.flags)

[array([[1, 1, 2, 2],
       [3, 3, 4, 4]]), array([[5, 6, 5, 6],
       [8, 7, 8, 7]])]
[[1 2 5 6 7]
 [1 2 8 7 6]
 [3 4 5 6 7]
 [3 4 8 7 6]]


In [48]:
rng = np.random.default_rng(0)
a = rng.random((3,3,3))
print(a)
a = np.swapaxes(a, 0,2)
print(a.flags)
a = np.ascontiguousarray(a)

a = a.reshape(-1, a.shape[-1])
print(a)
print(a.flags)

[[[0.63696169 0.26978671 0.04097352]
  [0.01652764 0.81327024 0.91275558]
  [0.60663578 0.72949656 0.54362499]]

 [[0.93507242 0.81585355 0.0027385 ]
  [0.85740428 0.03358558 0.72965545]
  [0.17565562 0.86317892 0.54146122]]

 [[0.29971189 0.42268722 0.02831967]
  [0.12428328 0.67062441 0.64718951]
  [0.61538511 0.38367755 0.99720994]]]
  C_CONTIGUOUS : False
  F_CONTIGUOUS : True
  OWNDATA : False
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False

[[0.63696169 0.93507242 0.29971189]
 [0.01652764 0.85740428 0.12428328]
 [0.60663578 0.17565562 0.61538511]
 [0.26978671 0.81585355 0.42268722]
 [0.81327024 0.03358558 0.67062441]
 [0.72949656 0.86317892 0.38367755]
 [0.04097352 0.0027385  0.02831967]
 [0.91275558 0.72965545 0.64718951]
 [0.54362499 0.54146122 0.99720994]]
  C_CONTIGUOUS : True
  F_CONTIGUOUS : False
  OWNDATA : False
  WRITEABLE : True
  ALIGNED : True
  WRITEBACKIFCOPY : False



In [51]:
a = rng.random((3,3,2))
b = rng.random((4,2,5))

np.tensordot(a,b, axes=(-1,-2)).shape

(3, 3, 4, 5)

In [4]:
import numpy as np

a = np.random.rand(5,3)

u,s,v = np.linalg.svd(a ,full_matrices=False)
print(u)
print(s)
print(v)

[[-0.44828432  0.88340502  0.07851437]
 [-0.64699997 -0.32102064 -0.23601873]
 [-0.50788052 -0.25073646  0.58562188]
 [-0.21315673  0.01502073 -0.76934059]
 [-0.27758564 -0.23118589 -0.05738217]]
[1.88315173 0.41726392 0.24263481]
[[-0.65552999 -0.5377234  -0.53022068]
 [ 0.00557885  0.69865428 -0.71543768]
 [-0.75514853  0.47194888  0.45498896]]


In [27]:
import numpy as np

a = np.arange(8).reshape(2,2,2)
b = np.arange(1,19).reshape(3,3,2)
print(a)
print(b)
ref = np.zeros((2,3,2,3,2,2))

for i in range(2):
  for j in range(3):
    for k in range(2):
      for l in range(3):
        for m in range(2):
          for n in range(2):
            ref[i,j,k,l,m,n] = a[i,k,m] * b[j,l,n] 

print(ref.reshape(6,6,4))
print(np.tensordot(a,b,axes=0).transpose([0,3,1,4,2,5]).reshape(6,6,4))


[[[0 1]
  [2 3]]

 [[4 5]
  [6 7]]]
[[[ 1  2]
  [ 3  4]
  [ 5  6]]

 [[ 7  8]
  [ 9 10]
  [11 12]]

 [[13 14]
  [15 16]
  [17 18]]]
[[[  0.   0.   1.   2.]
  [  0.   0.   3.   4.]
  [  0.   0.   5.   6.]
  [  2.   4.   3.   6.]
  [  6.   8.   9.  12.]
  [ 10.  12.  15.  18.]]

 [[  0.   0.   7.   8.]
  [  0.   0.   9.  10.]
  [  0.   0.  11.  12.]
  [ 14.  16.  21.  24.]
  [ 18.  20.  27.  30.]
  [ 22.  24.  33.  36.]]

 [[  0.   0.  13.  14.]
  [  0.   0.  15.  16.]
  [  0.   0.  17.  18.]
  [ 26.  28.  39.  42.]
  [ 30.  32.  45.  48.]
  [ 34.  36.  51.  54.]]

 [[  4.   8.   5.  10.]
  [ 12.  16.  15.  20.]
  [ 20.  24.  25.  30.]
  [  6.  12.   7.  14.]
  [ 18.  24.  21.  28.]
  [ 30.  36.  35.  42.]]

 [[ 28.  32.  35.  40.]
  [ 36.  40.  45.  50.]
  [ 44.  48.  55.  60.]
  [ 42.  48.  49.  56.]
  [ 54.  60.  63.  70.]
  [ 66.  72.  77.  84.]]

 [[ 52.  56.  65.  70.]
  [ 60.  64.  75.  80.]
  [ 68.  72.  85.  90.]
  [ 78.  84.  91.  98.]
  [ 90.  96. 105. 112.]
  [102. 108. 119. 

In [29]:
indices_interleave = np.empty(2 * a.ndim, dtype=int)
indices_interleave[::2] = np.arange(a.ndim)
indices_interleave[1::2] = np.arange(a.ndim, 2* a.ndim)
indices_interleave

array([0, 3, 1, 4, 2, 5])

In [31]:
(3,4) * (2,3)

TypeError: can't multiply sequence by non-int of type 'tuple'

In [39]:
# import copy

a= [np.random.rand(3)]

b = a.copy()

del b[0]

print(a)
print(b)

[array([0.08283985, 0.80598108, 0.43629905])]
[]


In [47]:
import numbers

isinstance(np.array(1), numbers.Number)
float(np.array(1))

1.0

In [67]:
import numpy as np

# a = np.arange(8).reshape(2,2,2)
# b = np.arange(1,19).reshape(3,3,2)
a = np.random.rand(2,2,2)
b = np.random.rand(3,3,2)
print(a)
print(b)
res = np.zeros((5,5,4))

slices = [slice(-d, None) for d in b.shape]
print(slices)
res[*slices] = b

res = res.reshape(5,5,4)


res[3,3,3], b[1,1,1]

[[[0.93278848 0.92161903]
  [0.06209288 0.01455047]]

 [[0.9490889  0.14792801]
  [0.98813679 0.84342615]]]
[[[0.37854694 0.5997749 ]
  [0.04356495 0.38378457]
  [0.34451313 0.19556269]]

 [[0.33303897 0.0628011 ]
  [0.81640293 0.95724354]
  [0.88307999 0.42192361]]

 [[0.8067145  0.4602535 ]
  [0.38058112 0.94677919]
  [0.79051945 0.03865877]]]
[slice(-3, None, None), slice(-3, None, None), slice(-2, None, None)]


(np.float64(0.9572435441023789), np.float64(0.9572435441023789))

In [5]:
import numpy as np

np.zeros(1) + np.arange(3)

array([0., 1., 2.])